In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/raesiow/23-nova-both-pairs/__notebook__.ipynb
/kaggle/input/notebooks/raesiow/23-nova-both-pairs/__output__.json
/kaggle/input/notebooks/raesiow/23-nova-both-pairs/__results__.html
/kaggle/input/notebooks/raesiow/23-nova-both-pairs/custom.css
/kaggle/input/notebooks/raesiow/09-build-index/__huggingface_repos__.json
/kaggle/input/notebooks/raesiow/09-build-index/__notebook__.ipynb
/kaggle/input/notebooks/raesiow/09-build-index/__output__.json
/kaggle/input/notebooks/raesiow/09-build-index/__results__.html
/kaggle/input/notebooks/raesiow/09-build-index/custom.css
/kaggle/input/notebooks/raesiow/09-build-index/index_products.csv
/kaggle/input/notebooks/raesiow/09-build-index/off_image_embeddings.npy
/kaggle/input/notebooks/raesiow/09-build-index/off_image_embeddings_altered.npy
/kaggle/input/notebooks/raesiow/09-build-index/retrieval_results.csv
/kaggle/input/datasets/raesiow/catalogue-predictions-v3/catalogue_predictions_v3.csv
/kaggle/input/datasets/raesiow/catal

In [2]:
folder = "/kaggle/input/notebooks/raesiow/09-build-index/"  # the 09 output folder
embeddings = np.load(folder + "off_image_embeddings.npy")  # one row of 384 numbers per product picture
products = pd.read_csv(folder + "index_products.csv", dtype={"code": str})  # product list, barcodes kept as text
print("embeddings shape:", embeddings.shape)  # expect (71241, 384)
print("same length:", len(products) == embeddings.shape[0])  # must be True

embeddings shape: (71241, 384)
same length: True


In [3]:
!pip install -q faiss-cpu easyocr  # search library and text reader (Internet must be ON)

In [4]:
import faiss  # fast nearest-neighbour search
import torch  # deep learning library
from transformers import AutoImageProcessor, AutoModel  # DINOv2 loader
from PIL import Image, ImageOps  # open images, apply phone rotation flag

index = faiss.IndexFlatIP(embeddings.shape[1])  # exact search by inner product
index.add(embeddings)  # put all vectors into the index
device = "cuda" if torch.cuda.is_available() else "cpu"  # use the GPU if there is one
print("vectors in index:", index.ntotal, "| device:", device)  # expect 71241 and cuda
processor = AutoImageProcessor.from_pretrained("facebook/dinov2-small")  # resizes and normalises images
dino = AutoModel.from_pretrained("facebook/dinov2-small").to(device)  # the model itself
dino.eval()  # evaluation mode

def embed_image(img):  # turn one PIL image into one 384-number vector
    inputs = processor(images=img.convert("RGB"), return_tensors="pt").to(device)  # resize, normalise, to GPU
    with torch.no_grad():  # no gradients needed
        out = dino(**inputs)  # run DINOv2
    vec = torch.nn.functional.normalize(out.pooler_output, dim=1)  # image summary vector, length 1
    return vec.cpu().numpy().astype("float32")  # back to numpy

vectors in index: 71241 | device: cuda


preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

In [5]:
import re  # splitting text into words
from collections import Counter  # counting words
import easyocr  # text reader

reader = easyocr.Reader(["en"], gpu=True)  # English reader on the GPU

def words_of(text):  # the set of lowercase words of 3+ letters
    return set(re.findall(r"[a-z]{3,}", str(text).lower()))  # letters only

name_words = products["product_name"].apply(words_of)  # word set of every catalogue name
doc_count = Counter(w for ws in name_words for w in ws)  # how many names contain each word
idf = {w: np.log(len(products) / c) for w, c in doc_count.items()}  # rare words get bigger values

def read_pack_text(img, angles):  # read the front text at each allowed angle, keep the best angle
    best_angle, best_score, best_text = angles[0], -1, ""  # nothing picked yet
    for angle in angles:  # each angle we are allowed to try
        pieces = reader.readtext(np.array(img.rotate(angle, expand=True)))  # (box, text, confidence) pieces
        good = [(t, c) for box, t, c in pieces if c > 0.5]  # keep pieces the reader is fairly sure about
        score = sum(c for t, c in good)  # total confidence at this angle
        if score > best_score:  # better than what we have so far
            best_angle, best_score, best_text = angle, score, " ".join(t for t, c in good)  # remember it
    return best_angle, best_text  # the picked angle and its text

rows_of_word = {}  # word -> list of catalogue rows whose name contains it
for row_number, ws in enumerate(name_words):  # every catalogue name
    for w in ws:  # every word in it
        rows_of_word.setdefault(w, []).append(row_number)  # remember the row
rows_of_word = {w: np.array(r) for w, r in rows_of_word.items()}  # lists to arrays for speed

In [6]:
import os  # file listing
import requests  # download pictures
from io import BytesIO  # treat bytes like a file
from sentence_transformers import SentenceTransformer  # the MiniLM text model

cat_path = None  # not found yet
for folder_, _, names in os.walk("/kaggle/input"):  # look through every input folder
    for name in names:  # each file
        if name == "catalogue_predictions_v3.csv":  # the file we want
            cat_path = os.path.join(folder_, name)  # remember the path
print("catalogue file:", cat_path)  # show it
cat = pd.read_csv(cat_path, dtype={"code": str})  # load it, barcodes as text
band_rank = {"low": 0, "medium": 1, "high": 2}  # lower is better
cat["band_rank"] = cat["gi_band_new"].map(band_rank)  # band words to numbers
cat.loc[cat["gi_not_estimated"], "band_rank"] = 1  # hidden GI counts as medium, for ranking only
cat["gi_number_rank"] = cat["gi_number_new"].clip(0, 100)  # GI number clipped to 0-100
cat.loc[cat["gi_not_estimated"], "gi_number_rank"] = 50  # hidden GI counts as 50, for ranking only

mini = SentenceTransformer("all-MiniLM-L6-v2", device=device)  # MiniLM on the GPU
name_vecs = mini.encode(cat["product_name"].fillna("").tolist(), batch_size=256,  # every product name as numbers
                        normalize_embeddings=True, show_progress_bar=False)  # length 1
print("name vectors:", name_vecs.shape)  # expect (71298, 384)

catalogue file: /kaggle/input/datasets/raesiow/catalogue-predictions-v3/catalogue_predictions_v3.csv


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

name vectors: (71298, 384)


In [7]:
def picture_first(img, text, n=5, K=100):  # row A: shortlist K by picture, re-rank by shared words
    img = ImageOps.exif_transpose(img).convert("RGB")  # apply the phone's rotation flag
    sims, rows = index.search(embed_image(img), K)  # the K most picture-similar products
    ocr_words = words_of(text)  # words read from the pack
    scored = []  # (word score, similarity, row) for each shortlisted product
    for sim, row in zip(sims[0], rows[0]):  # each shortlisted product
        matched = ocr_words & name_words[row]  # words shared with the pack text
        scored.append((sum(idf[w] for w in matched), float(sim), int(row)))  # score = sum of rarity values
    scored.sort(key=lambda s: (s[0], s[1]), reverse=True)  # best word score first, ties by picture
    return [r for s, sim, r in scored[:n]]  # row numbers of the top n

def final_candidates(img, n=5, n_from_a=3):  # 3 from row A, then row B, no duplicates, 5 in total
    img = ImageOps.exif_transpose(img).convert("RGB")  # apply the phone's rotation flag
    small = img.copy()  # a copy for the text reader
    small.thumbnail((1600, 1600))  # shrink it to at most 1600 px
    _, text = read_pack_text(small, [0])  # read the pack text once, upright
    list_a = picture_first(img, text, n=10)  # row A, best first (10 kept as reserve)
    word_score = np.zeros(len(products))  # score of every catalogue product
    for w in words_of(text):  # each word read from the pack
        if w in rows_of_word:  # only words that appear in some name
            word_score[rows_of_word[w]] += idf[w]  # add the rarity value
    similarity = (embeddings @ embed_image(img)[0]).astype("float64")  # picture similarity to every product
    list_b = np.lexsort((similarity, word_score))[::-1][:10].tolist()  # row B, best first (10 kept)
    chosen = [(r, "A") for r in list_a[:n_from_a]]  # first the top 3 of row A
    for r in list_b:  # then row B, best first
        if len(chosen) >= n:  # stop at 5
            break
        if r not in [c[0] for c in chosen]:  # skip duplicates
            chosen.append((r, "B"))  # remember the source
    for r, src in [(r, "A") for r in list_a] + [(r, "B") for r in list_b]:  # fill any gaps from the reserves
        if len(chosen) >= n:  # stop at 5
            break
        if r not in [c[0] for c in chosen]:  # skip duplicates
            chosen.append((r, src))  # remember the source
    return chosen, text  # list of (row, source) and the text read

def recommend3(code, n=3, pool=200, min_similarity=0.5):  # chosen barcode in; alternatives out
    positions = np.where(cat["code"].values == str(code))[0]  # row position of the chosen product
    own_pos = positions[0]  # position of the chosen product
    own = cat.iloc[own_pos]  # its row
    category = own["pnns_groups_2"]  # its food category
    if pd.isna(category) or str(category).lower() == "unknown":  # no usable category
        return own, "no same-category alternative available"  # as agreed
    same_pos = np.where((cat["pnns_groups_2"].values == category) & (np.arange(len(cat)) != own_pos))[0]  # same category, not itself
    sims = name_vecs[same_pos] @ name_vecs[own_pos]  # name similarity to the chosen product
    keep = same_pos[np.argsort(-sims)[:pool]]  # the pool most similar by name
    similar = cat.iloc[keep].copy()  # their rows
    similar["name_similarity"] = np.sort(sims)[::-1][:pool].round(2)  # remember the similarity
    similar = similar[similar["name_similarity"] >= min_similarity]  # the guard: drop products with unlike names
    lower_nova = similar["nova_pred"] < own["nova_pred"]  # strictly lower NOVA
    if own["gi_not_estimated"]:  # the chosen product's GI is hidden
        better = similar[lower_nova]  # only NOVA can count
    else:  # GI is shown
        lower_gi = ((similar["nova_pred"] == own["nova_pred"]) & (~similar["gi_not_estimated"]) &  # same NOVA, GI shown
                    (similar["gi_number_rank"] <= own["gi_number_rank"] - 10))  # and GI at least 10 lower
        better = similar[lower_nova | lower_gi]  # either way counts
    better = better.sort_values(["nova_pred", "gi_number_rank"])  # lowest NOVA, then lowest GI number
    if len(better) == 0:  # nothing better and similar
        return own, "no similar product with a better rating found"  # say so honestly
    return own, better.head(n)  # the chosen product and its top alternatives
print("ready")  # done

ready


In [8]:
import gradio as gr  # the app library

code_to_pos = {c: i for i, c in enumerate(cat["code"])}  # barcode -> row position in the catalogue table

def picture_of(code):  # the catalogue picture of a product, or a white square if it fails
    url = cat.iloc[code_to_pos[code]]["image_url"]  # picture web address
    try:  # the download might fail
        return Image.open(BytesIO(requests.get(url, timeout=20).content)).convert("RGB")  # download and open it
    except Exception:
        return Image.new("RGB", (200, 200), "white")  # placeholder

def to_gallery(codes):  # list of (picture, caption) for the gallery
    return [(picture_of(c), cat.iloc[code_to_pos[c]]["product_name"][:40]) for c in codes]  # one per product

def find_candidates(photo):  # photo in -> 5 candidates out
    if photo is None:  # nothing uploaded
        return [], [], "No photo yet."  # tell the user
    chosen, text = final_candidates(photo)  # the fixed 3+2 rule
    codes = [products.loc[r, "code"] for r, src in chosen]  # barcodes of the candidates
    codes = [c for c in codes if c in code_to_pos]  # keep those that are in the catalogue table
    return to_gallery(codes), codes, "Text read from the pack: " + text[:120]  # gallery, codes, text

def search_by_name(query):  # typed name in -> 5 most similar product names out
    vec = mini.encode([query], normalize_embeddings=True)[0]  # turn the typed text into numbers
    top = np.argsort(-(name_vecs @ vec))[:5]  # the 5 most similar names
    codes = cat["code"].iloc[top].tolist()  # their barcodes
    return to_gallery(codes), codes, "Search by name: " + query  # gallery, codes, text

def similar_listings(code, threshold=0.9):  # all listings whose names are almost the same as this one
    pos = np.where(cat["code"].values == str(code))[0][0]  # row position of the chosen product
    sims = name_vecs @ name_vecs[pos]  # name similarity to every product
    return cat.iloc[np.where(sims >= threshold)[0]]  # the listings at or above the cutoff (includes itself)

def band_of_number(x):  # the GI band for a number
    return "low" if x <= 55 else ("medium" if x <= 69 else "high")  # low <= 55, medium 56-69, high >= 70

def gi_phrase(band, number):  # one GI phrase from a band and a number
    number = float(min(max(number, 0), 100))  # clip to 0-100
    if band == band_of_number(number):  # the two estimates agree
        return f"{band} (approx. {number:.0f})"  # band and number
    return f"uncertain: band model says {band}, number estimate about {number:.0f} ({band_of_number(number)})"  # show both, say they disagree

def gi_text(row):  # GI text for one product (used for the alternatives)
    if row["gi_not_estimated"]:  # hidden for low-carb foods and sauces
        return "not estimated"  # say so
    return gi_phrase(row["gi_band_new"], row["gi_number_new"])  # band and number, or both with a warning

def describe(codes, evt: gr.SelectData):  # the user tapped a candidate -> show NOVA, GI, ingredients and alternatives
    code = codes[evt.index]  # barcode of the tapped product
    own, result = recommend3(code)  # the tapped listing and its alternatives
    group = similar_listings(code)  # listings with almost the same name
    n = len(group)  # how many were pooled
    nova = int(group["nova_pred"].mode().iloc[0])  # most common NOVA group
    conf = group["nova_confidence"].median()  # typical confidence
    if group["gi_not_estimated"].mean() > 0.5:  # GI hidden for most of them
        gi_line = "not estimated"  # say so
    else:  # GI is shown
        shown = group[~group["gi_not_estimated"]]  # listings with a GI
        gi_line = gi_phrase(shown["gi_band_new"].mode().iloc[0], shown["gi_number_new"].median())  # most common band, median number
    text = ingredient_of.get(code)  # the tapped listing's own ingredient text
    has_text = isinstance(text, str) and text.strip() != ""  # True if there is real ingredient text
    ingredients = str(text).replace("\n", " ")[:250] if has_text else "not available"  # first 250 characters
    additives = additive_line(text) if has_text else "Additive types listed: not available"  # counted from the FULL text
    lines = [f"### {own['product_name']}",  # product name
             f"**Processing level (NOVA):** group {nova} (model confidence {conf:.0%})",  # NOVA
             f"**Glycaemic index:** {gi_line}",  # GI
             f"**Ingredients (from catalogue):** {ingredients}",  # what the prediction was based on
             f"**{additives}**"]  # additive types found in the ingredient text
    if n > 1:  # more than one listing was pooled
        lines.append(f"*NOVA and GI based on {n} listings with very similar names.*")  # tell the user
    lines.append("\n**Possible alternatives (based on predicted NOVA and GI):**")  # heading
    if isinstance(result, str):  # a message instead of a table
        lines.append("- " + result)  # print it
    else:  # a table of alternatives
        for _, r in result.iterrows():  # each alternative
            lines.append(f"- {r['product_name']} (NOVA {int(r['nova_pred'])}, GI {gi_text(r)})")  # one line each
    lines.append("\n*Model estimates for guidance only, not medical advice.*")  # the caveat
    return "\n".join(lines)  # one text block
print("app helpers ready")  # done

app helpers ready


In [9]:
import re  # regular expressions: lets us search text for word patterns

# each additive type and the pattern that finds it in ingredient text
# (s? = optional plural, [sz] = both spellings, ou? = colour or color)
additive_patterns = {
    "emulsifier": r"\bemulsifiers?\b",                    # e.g. emulsifier (471)
    "thickener": r"\bthickeners?\b",                      # e.g. thickener (1442)
    "stabiliser": r"\bstabili[sz]ers?\b",                 # stabiliser or stabilizer
    "flavour enhancer": r"\bflavou?r enhancers?\b",       # e.g. flavour enhancer (621)
    "flavour": r"\bflavou?r(ing)?s?\b(?! enhancer)",      # flavour, but not "flavour enhancer"
    "colour": r"\bcolou?r(ing)?s?\b",                     # colour, color, colouring
    "preservative": r"\bpreservatives?\b",                # e.g. preservative (202)
    "antioxidant": r"\bantioxidants?\b",                  # e.g. antioxidant (307b)
    "acidity regulator": r"\bacidity regulators?\b",      # e.g. acidity regulator (330)
    "food acid": r"\bfood acids?\b",                      # Australian label wording
    "mineral salt": r"\bmineral salts?\b",                # e.g. mineral salt (451)
    "raising agent": r"\braising agents?\b",              # e.g. raising agents (500)
    "humectant": r"\bhumectants?\b",                      # keeps food moist
    "sweetener": r"\bsweeteners?\b",                      # e.g. sweetener (955)
    "gelling agent": r"\bgelling agents?\b",              # e.g. gelling agent (440)
    "anti-caking agent": r"\banti-?caking agents?\b",     # anticaking or anti-caking
    "glazing agent": r"\bglazing agents?\b",              # shiny coating
    "firming agent": r"\bfirming agents?\b",              # e.g. in canned vegetables
}

def additive_types_in(text):                               # finds which additive types appear in one ingredient text
    text = str(text).lower()                               # make text lowercase (and turn missing values into a string)
    found = []                                             # empty list to collect the types we find
    for name, pattern in additive_patterns.items():        # go through each additive type and its pattern
        if re.search(pattern, text):                       # if the pattern appears anywhere in the text
            found.append(name)                             # remember this type
    return found                                           # give back the list of types found

def additive_line(text):                                   # turns the list into one sentence for the app
    found = additive_types_in(text)                        # find the types in this text
    if len(found) == 0:                                    # if none were found
        return "Additive types listed: none found"         # say so
    return f"Additive types listed: {len(found)} ({', '.join(found)})"  # e.g. "3 (emulsifier, flavour, colour)"

# quick test on two made-up ingredient lists
print(additive_line("Wheat flour, sugar, emulsifier (471), flavour enhancer (621), natural flavour, colour (160b)"))  # expect 4
print(additive_line("Rolled oats"))                        # expect none found

Additive types listed: 4 (emulsifier, flavour enhancer, flavour, colour)
Additive types listed: none found


In [10]:
off_path = None  # not found yet
for folder_, _, names in os.walk("/kaggle/input"):  # look through every input folder
    for name in names:  # each file
        if name == "off_filtered_all_countries.csv":  # the OFF file with ingredient text
            off_path = os.path.join(folder_, name)  # remember the path
off_text = pd.read_csv(off_path, usecols=["code", "ingredients_text"], dtype={"code": str})  # only two columns
ingredient_of = dict(zip(off_text["code"], off_text["ingredients_text"]))  # barcode -> ingredient text
print(len(ingredient_of), "ingredient texts loaded")  # expect about 376,917

376917 ingredient texts loaded


In [11]:
try:  # close an older running app, if any
    demo.close()  # stops it
except Exception:
    pass  # nothing to close

with gr.Blocks() as demo:  # the app layout
    gr.Markdown("# Food Scanner\nPhoto of the front of the pack, held upright.")  # title
    photo = gr.Image(type="pil", label="Photo of the product", sources=["upload", "webcam"], webcam_options=gr.WebcamOptions(mirror=False))  # upload and camera, camera not mirrored
    find_btn = gr.Button("Find product")  # start button
    read_text = gr.Markdown()  # shows the text read from the pack
    gallery = gr.Gallery(label="Is your product here? Tap it.", columns=5, height=260, allow_preview=False)  # the 5 candidates
    codes_state = gr.State([])  # remembers the candidates' barcodes
    query = gr.Textbox(label="Not here? Type the product name")  # fallback box
    search_btn = gr.Button("Search by name")  # fallback button
    result_md = gr.Markdown()  # NOVA, GI and alternatives appear here
    find_btn.click(find_candidates, photo, [gallery, codes_state, read_text])  # photo -> candidates
    search_btn.click(search_by_name, query, [gallery, codes_state, read_text])  # typed name -> candidates
    gallery.select(describe, codes_state, result_md)  # tap -> results

demo.launch(share=True)  # start the app and print a link

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://3017508422c142a266.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
